# 01 Data preparation

Processing xlsx files from data folder into suitable inputs and generate other input files

In [1]:
import pandas as pd
import numpy as np
import os
import pickle

In [2]:
cell_line ='BC3C_dec'
data_dir = f"/home/jing/Phd_project/project_UCD_blca/blca_publication_OUTPUT/blca_publication_OUTPUT_LINCS/00_outputs_2020_{cell_line}/"
info_dir = data_dir
out_dir = f"/home/jing/Phd_project/project_UCD_blca/blca_publication_OUTPUT/blca_publication_OUTPUT_bmra/blca_publication_OUTPUT_bmra_{cell_line}/00_outputs_2020_{cell_line}/"

#os.makedirs(info_dir, exist_ok = True)
os.makedirs(out_dir, exist_ok = True)

## Modules

Load data about modules and drugs.

In [3]:
os.path.join(info_dir, "ALL_DATA_2020_post.xlsx")

'/home/jing/Phd_project/project_UCD_blca/blca_publication_OUTPUT/blca_publication_OUTPUT_LINCS/00_outputs_2020_BC3C_dec/ALL_DATA_2020_post.xlsx'

In [4]:
### DATA
### remove whitespaces in names (modules), remove duplicates

modules_df = pd.read_excel(
    os.path.join(info_dir, "ALL_DATA_2020_post.xlsx"), sheet_name = "modules", index_col = 0)
# display(modules_df)

selected_modules = modules_df.index.tolist()
print(len(selected_modules), ' - Size after reading')

# remove duplicates
selected_modules = modules_df.index.unique().tolist()
print(len(selected_modules), ' - Size after remove duplicates')

# remove whitespaces in modules' names 
selected_modules = [d.strip() for d in selected_modules]

print('Selected modules list: ', len(selected_modules), selected_modules)


10  - Size after reading
10  - Size after remove duplicates
Selected modules list:  10 ['CDK1_2', 'CDK4_6', 'EGFR', 'Estrogen', 'FGFR', 'PI3K', 'p53', 'TOP2A', 'Src', 'SMAD3']


In [5]:
### DATA
### remove whitespaces in names (modules, drugs), remove duplicates
### check the dimensions of the indicator IC50 1 uM = 1000 nM
### copy-paste as values, numbers, no formulas

IC50_df = pd.read_excel(
    os.path.join(info_dir, "ALL_DATA_2020_post.xlsx"), sheet_name = "IC50s")
IC50_df.drop(columns=['Unnamed: 3'],inplace=True)

print(len(IC50_df.index), ' - Size after reading')
# display(IC50_df)

# rename
IC50_df = IC50_df.rename(columns = {"IC50, uM": "IC50"})

# manually correcting value IC50, Example  for IOX2 -> 30 nM  
#IC50_df.loc[IC50_df.index == 'IOX2', IC50_df.columns == 'IC50'] = 30/1000

# remove non-selected modules, modules' names with whitespaces or empty
#IC50_df = IC50_df[IC50_df.Module.isin(selected_modules)]

#print(len(IC50_df.index), ' - Size after remove modules')
# display(IC50_df)

# remove duplicates 
# Considering certain columns is optional. 
# Indexes, including time indexes are ignored.
IC50_df = IC50_df.drop_duplicates()

print(len(IC50_df.index), ' - Size after remove duplicates')
display(IC50_df)

33  - Size after reading
33  - Size after remove duplicates


,Drug,Module,IC50
0,JNJ-7706621,CDK1_2,0.02700
1,PHA-793887,CDK1_2,0.18000
2,roscovitine,CDK1_2,2.00000
3,alvocidib,CDK4_6,0.12000
4,palbociclib,CDK4_6,0.04500
5,afatinib,EGFR,0.03000
6,erlotinib,EGFR,0.00600
7,gefitinib,EGFR,0.17100
8,lapatinib,EGFR,0.03240
9,vandetanib,EGFR,1.50000


In [6]:
modules = IC50_df.Module.unique().tolist()

print('IC50_df  modules list: ', len(modules), modules)
print()
print('Selected modules list: ', len(selected_modules), selected_modules)

### CHECK
print()
print('CHECK: ', len(selected_modules),'=?', len(modules))

n_modules = len(modules)


IC50_df  modules list:  10 ['CDK1_2', 'CDK4_6', 'EGFR', 'Estrogen', 'FGFR', 'PI3K', 'p53', 'TOP2A', 'Src', 'SMAD3']

Selected modules list:  10 ['CDK1_2', 'CDK4_6', 'EGFR', 'Estrogen', 'FGFR', 'PI3K', 'p53', 'TOP2A', 'Src', 'SMAD3']

CHECK:  10 =? 10


In [7]:
drugs = IC50_df.Drug.tolist()
print(len(drugs), ' - Size after reading')

# remove duplicates
drugs = IC50_df.Drug.unique().tolist()
print(len(drugs), ' - Size after remove duplicates')

# remove whitespaces in drugs' names (necessary for some)
drugs = [d.strip() for d in drugs]

# remove duplicates after remove whitespaces
drugs = list(set(drugs))
print(len(drugs), ' - Size after remove duplicates without whitespaces')

print('Drugs list: ', len(drugs), drugs)

n_drugs = len(drugs)

33  - Size after reading
33  - Size after remove duplicates
33  - Size after remove duplicates without whitespaces
Drugs list:  33 ['dasatinib', 'RITA', 'daunorubicin', 'afatinib', 'serdemetan', 'vandetanib', 'AS-605240', 'SAR405838', 'taselisib', 'estradiol-cypionate', 'epirubicin', 'palbociclib', 'HLI-373', 'alvocidib', 'dienestrol', 'sorafenib', 'roscovitine', 'gefitinib', 'nutlin-3', 'masitinib', 'PI-103', 'ponatinib', 'idarubicin', 'erlotinib', 'JNJ-7706621', 'mitoxantrone', 'lapatinib', 'LY-294002', 'Agent2', 'raloxifene', 'AMG-232', 'AZD-8055', 'PHA-793887']


## L1000 meta data

Get sig_id for selected drugs.

In [8]:
sig_info_df = pd.read_excel(os.path.join(data_dir, f"sig_info_2020_BC3C.xlsx"), index_col = 0)

display(sig_info_df)

,cell,plate,time,level_3_samples,samples_number,pert_type,pert_drug,targets,targets_number,dose,dose_float
level_5_sig_id,,,,,,,,,,,
ASG002_BC3C_24H:A03,BC3C,ASG002,24 h,ASG002_BC3C_24H_X1_B35:A03,1,ctl_vehicle,DMSO,DMSO_No_target,0,0 uM,0.00
ASG002_BC3C_24H:A04,BC3C,ASG002,24 h,ASG002_BC3C_24H_X1_B35:A04,1,ctl_vehicle,DMSO,DMSO_No_target,0,0 uM,0.00
ASG002_BC3C_24H:A05,BC3C,ASG002,24 h,ASG002_BC3C_24H_X1_B35:A05,1,ctl_vehicle,DMSO,DMSO_No_target,0,0 uM,0.00
ASG002_BC3C_24H:A06,BC3C,ASG002,24 h,ASG002_BC3C_24H_X1_B35:A06,1,ctl_vehicle,DMSO,DMSO_No_target,0,0 uM,0.00
ASG002_BC3C_24H:J13,BC3C,ASG002,24 h,ASG002_BC3C_24H_X1_B35:J13,1,ctl_vehicle,DMSO,DMSO_No_target,0,0 uM,0.00
...,...,...,...,...,...,...,...,...,...,...,...
MOAR012_BC3C_24H:P20,BC3C,MOAR012,24 h,MOAR012_BC3C_24H_X1_B36:P20,1,trt_cp,BAY-61-3606,NaN,0,3.33 uM,3.33
MOAR012_BC3C_24H:P21,BC3C,MOAR012,24 h,MOAR012_BC3C_24H_X1_B36:P21,1,trt_cp,BAY-61-3606,NaN,0,1.11 uM,1.11
MOAR012_BC3C_24H:P22,BC3C,MOAR012,24 h,MOAR012_BC3C_24H_X1_B36:P22,1,trt_cp,ethaverine,NaN,0,10 uM,10.00


In [9]:
# now filtering so only the required drugs are present
sig_info_df = sig_info_df.loc[sig_info_df.pert_drug.isin(drugs)]

# here's what we have now
display(sig_info_df)

,cell,plate,time,level_3_samples,samples_number,pert_type,pert_drug,targets,targets_number,dose,dose_float
level_5_sig_id,,,,,,,,,,,
ASG002_BC3C_24H:A10,BC3C,ASG002,24 h,ASG002_BC3C_24H_X1_B35:A10,1,trt_cp,taselisib,PIK3CA,1,10 uM,10.00
ASG002_BC3C_24H:A11,BC3C,ASG002,24 h,ASG002_BC3C_24H_X1_B35:A11,1,trt_cp,taselisib,PIK3CA,1,1.11 uM,1.11
ASG002_BC3C_24H:A19,BC3C,ASG002,24 h,ASG002_BC3C_24H_X1_B35:A19,1,trt_cp,AS-605240,PIK3CG,1,10 uM,10.00
ASG002_BC3C_24H:A20,BC3C,ASG002,24 h,ASG002_BC3C_24H_X1_B35:A20,1,trt_cp,AS-605240,PIK3CG,1,1.11 uM,1.11
ASG002_BC3C_24H:A21,BC3C,ASG002,24 h,ASG002_BC3C_24H_X1_B35:A21,1,trt_cp,AS-605240,PIK3CG,1,0.12 uM,0.12
...,...,...,...,...,...,...,...,...,...,...,...
MOAR010_BC3C_24H:K08,BC3C,MOAR010,24 h,MOAR010_BC3C_24H_X1_B36:K08,1,trt_cp,SAR405838,MDM2,1,3.33 uM,3.33
MOAR010_BC3C_24H:K09,BC3C,MOAR010,24 h,MOAR010_BC3C_24H_X1_B36:K09,1,trt_cp,SAR405838,MDM2,1,1.11 uM,1.11
MOAR011_BC3C_24H:F07,BC3C,MOAR011,24 h,MOAR011_BC3C_24H_X1_B36:F07,1,trt_cp,serdemetan,MDM2,1,10 uM,10.00


In [10]:
#previous removed
#rows_remove = ['ASG002_BC3C_24H:F04', 'ASG002_BC3C_24H:F05','ASG002_BC3C_24H:L02',
#               'ASG002_BC3C_24H:O24','ASG002_B                         C3C_24H:L07','ASG002_BC3C_24H:L09',
#               'MOAR010_BC3C_24H:D02', 'ASG002_BC3C_24H:F01','MOAR010_BC3C_24H:D01',
#               'ASG002_BC3C_24H:N19','ASG002_BC3C_24H:N21','ASG002_BC3C_24H:N24',
#               'ASG002_BC3C_24H:I19','ASG002_BC3C_24H:I21','ASG002_BC3C_24H:L17',
#               'ASG002_BC3C_24H:M23','MOAR008_BC3C_24H:L03','MOAR010_BC3C_24H:L20','MOAR011_BC3C_24H:J10',
#               'MOAR008_BC3C_24H:L08','MOAR009_BC3C_24H:C10','MOAR010_BC3C_24H:A13','MOAR010_BC3C_24H:A14','MOAR011_BC3C_24H:F09',
#               'ASG002_BC3C_24H:G01','ASG002_BC3C_24H:P20']

In [11]:
rows_remove = ['ASG002_BC3C_24H:G01',
 'ASG002_BC3C_24H:F15',
 'ASG002_BC3C_24H:N19',
 'ASG002_BC3C_24H:N22',
 'ASG002_BC3C_24H:F15',
 #'ASG002_BC3C_24H:I08',
 'ASG002_BC3C_24H:I19',
 'ASG002_BC3C_24H:B12',
 'MOAR008_BC3C_24H:L07',
 'MOAR008_BC3C_24H:L09',
 'ASG002_BC3C_24H:F04',
 'ASG002_BC3C_24H:P17',
 'ASG002_BC3C_24H:O24',
 #'MOAR010_BC3C_24H:D02',
 'ASG002_BC3C_24H:L18',
 #'MOAR008_BC3C_24H:L03',
 #'MOAR010_BC3C_24H:L21'
 ]

Manually remove  few inhibition from data set, since it does differ from the other data points.

inhib_to_filter = "PF-03758309"
dose_to_filter = 10

id_to_filter = sig_info_df[
    np.logical_and(
        sig_info_df.pert_drug == inhib_to_filter,
        sig_info_df.dose_float == dose_to_filter,
    )
].index.values

print(id_to_filter)

sig_info_df = sig_info_df[~sig_info_df.index.isin(id_to_filter)]
display(sig_info_df)

inhib_to_filter = "roscovitine"
dose_to_filter = 3.33

id_to_filter = sig_info_df[
    np.logical_and(
        sig_info_df.pert_drug == inhib_to_filter,
        sig_info_df.dose_float == dose_to_filter,
    )
].index.values

print(id_to_filter)

sig_info_df = sig_info_df[~sig_info_df.index.isin(id_to_filter)]
display(sig_info_df)

In [12]:
sig_info_df.drop(rows_remove,inplace=True)

In [13]:
exp_ids = sig_info_df.index.unique().tolist()
print('Experiments ids list: ', len(exp_ids))

n_experiments = len(exp_ids)

Experiments ids list:  86


In [14]:
n_experiments = len(exp_ids)
print('Experiments ids list: ', len(exp_ids))

Experiments ids list:  86


Confirm data by checking the drugs of interest against the filtered L1000 meta data.

In [15]:
print(f"Number of drugs of interest:\t{len(drugs)}")
#print(f'Number of drugs in L1000 data:\t{len(sig_info_df.value_counts("drugs"))}')

#sig_info_df.value_counts("drugs")

Number of drugs of interest:	33


## L1000 data

In [16]:
Data_norm_df = pd.read_excel(os.path.join(data_dir, f"Data_norm_2020_BC3C.xlsx"), index_col = 0)
display(Data_norm_df)

,AARS,ABCB6,ABCC5,ABCF1,ABCF3,ABHD4,ABHD6,ABL1,ACAA1,ACAT2,...,ZMIZ1,ZMYM2,ZNF131,ZNF274,ZNF318,ZNF395,ZNF451,ZNF586,ZNF589,ZW10
ASG002_BC3C_24H:A03,-0.191254,-0.055246,0.039596,-0.256266,-0.040419,-0.590523,-0.159396,-0.074319,0.457981,0.409608,...,0.543203,0.494266,-0.011923,-0.225931,0.285054,-0.775246,0.166031,-0.024873,0.238723,0.284204
ASG002_BC3C_24H:A04,-0.265754,-0.317496,0.118696,-0.136665,-0.301569,-0.403023,0.124804,-0.036470,0.311931,0.660457,...,-0.565096,-0.088634,0.122977,-0.047931,0.141804,0.129054,-0.028819,-0.028773,-0.253627,-0.752646
ASG002_BC3C_24H:A05,-0.181954,-0.081597,-0.210304,1.559535,-0.019019,-0.457423,0.071404,0.074080,-0.356119,0.498808,...,0.226104,-0.228034,-0.121023,-0.075331,-0.133146,0.355054,0.022831,-0.084073,0.283123,-0.894896
ASG002_BC3C_24H:A06,0.033446,0.042404,-0.150154,-0.093165,0.053180,-0.053823,0.087704,0.167681,-0.601569,0.383308,...,-0.608596,-0.228835,0.072777,0.082970,-0.570996,2.847754,-0.211670,-0.067273,0.081723,0.338704
ASG002_BC3C_24H:J13,0.204446,0.180704,0.089096,-0.054666,0.053381,0.044877,-0.277396,-0.157419,0.535681,-3.933493,...,-0.318397,0.122265,-0.134323,-0.088931,-0.067996,-0.515847,-0.005069,0.067527,0.002223,0.204904
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
MOAR012_BC3C_24H:P20,0.647151,0.211700,-0.979200,0.597350,-0.375751,0.388300,0.394524,0.120151,-0.166775,-1.129125,...,-1.598475,-0.552750,0.515151,0.120800,0.082675,0.529700,0.383225,-0.207225,2.268450,-1.248500
MOAR012_BC3C_24H:P21,0.171800,0.046300,-0.145550,-0.295150,0.030849,0.420951,0.222075,0.179800,0.274724,-0.423975,...,-1.650575,0.203600,-0.003250,-0.064800,-0.037675,0.076499,0.201825,0.416875,0.287450,-0.971700
MOAR012_BC3C_24H:P22,0.648700,0.058749,-0.031700,0.408249,-0.753950,0.332200,-0.357525,-0.107650,-0.213575,0.074225,...,0.127625,0.031600,0.103250,-0.249600,0.046375,1.486200,0.440325,0.090075,-0.031650,-0.944300
MOAR012_BC3C_24H:P23,0.090499,-0.469300,-0.611800,0.873550,-0.788450,-0.097199,-0.366575,-0.490600,-0.624675,-0.009275,...,0.054676,-0.596050,0.084600,0.444700,0.431375,-0.921501,0.044926,0.716076,-0.000900,-1.106700


In [17]:
Data_norm_df = Data_norm_df[Data_norm_df.index.isin(exp_ids)]

# arrange experiments in same order as in list
Data_norm_df["sort_col"] = Data_norm_df.index.map({val: i for i, val in enumerate(exp_ids)})
Data_norm_df = Data_norm_df.sort_values("sort_col")
Data_norm_df = Data_norm_df.drop("sort_col", axis = 1)

# transpose
Data_norm_df = Data_norm_df.T

display(Data_norm_df)

,ASG002_BC3C_24H:A10,ASG002_BC3C_24H:A11,ASG002_BC3C_24H:A19,ASG002_BC3C_24H:A20,ASG002_BC3C_24H:A21,ASG002_BC3C_24H:B10,ASG002_BC3C_24H:B11,ASG002_BC3C_24H:B14,ASG002_BC3C_24H:B15,ASG002_BC3C_24H:C13,...,MOAR010_BC3C_24H:A15,MOAR010_BC3C_24H:J22,MOAR010_BC3C_24H:J23,MOAR010_BC3C_24H:J24,MOAR010_BC3C_24H:K07,MOAR010_BC3C_24H:K08,MOAR010_BC3C_24H:K09,MOAR011_BC3C_24H:F07,MOAR011_BC3C_24H:F08,MOAR011_BC3C_24H:F09
AARS,-0.496854,0.288446,0.189747,-0.016454,0.080746,0.282346,0.326246,0.303046,0.387546,0.434746,...,0.224383,-0.465317,0.006583,0.227732,1.393332,0.106531,-0.217618,0.584300,0.114500,-0.268751
ABCB6,-0.658596,-0.142196,-0.075397,-0.383796,-0.199996,-0.074197,0.108804,-0.399196,-0.227496,-0.041146,...,0.374653,-0.658898,-0.736548,0.386902,-0.729623,0.225201,0.202852,0.006950,-0.037150,-0.014150
ABCC5,-0.080204,0.231996,-0.329354,-0.225204,0.278446,0.034696,0.396596,-0.255704,-0.254154,-0.002004,...,0.083343,-0.365557,0.184543,0.096643,-0.388082,0.010793,0.011493,-0.238750,0.409550,0.221350
ABCF1,0.202535,0.602335,0.403335,0.313134,-0.083265,-0.056365,-0.387216,0.097235,0.540634,-0.262666,...,-1.130589,-0.940039,0.417610,-0.167539,0.231361,-0.288040,-0.458439,0.142675,0.077875,0.695325
ABCF3,-0.520919,-0.192819,0.001032,-0.096419,0.210881,-0.731118,0.095381,0.313481,-0.078018,-0.257669,...,0.404131,-0.375018,0.305533,-0.088618,0.119233,0.008182,0.201432,-0.468050,-0.140650,-0.353450
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
ZNF395,1.796254,1.773154,0.668354,0.825354,0.350654,0.205954,-0.998346,0.235354,0.230653,0.440854,...,-1.278881,-1.306905,0.159970,-1.107780,0.037869,0.367270,-0.285181,0.257250,-1.828550,-1.550550
ZNF451,-0.244519,0.116732,0.058081,-0.178169,-0.080619,-0.139119,0.064681,-0.083019,-0.272119,-0.164119,...,0.029882,-0.269068,0.067832,0.015482,-0.113718,0.029882,0.085807,0.080650,0.119950,0.044250
ZNF586,0.097627,0.061027,-0.337573,0.098427,-0.338173,0.017627,-0.197423,-0.078473,-0.300473,-0.147773,...,-0.090739,0.332686,-0.455289,-0.368264,0.090186,-0.182964,0.056085,-0.561900,0.202100,0.043000
ZNF589,0.608573,0.106123,-0.014126,-0.003677,-0.123477,-0.243377,0.044623,-0.021477,0.201823,-0.413227,...,-0.113816,0.402309,0.062659,-0.056591,-0.123891,-0.081541,-0.200241,-0.557600,-0.364600,-0.531500


### Concatenating TGFbRin and SMAD3in LFC2

In [18]:
files_path ='/home/jing/Phd_project/project_UCD_blca/blca_publication_OUTPUT/blca_publication_OUTPUT_bc3c_oct'

In [19]:
counts = pd.read_csv(
    '/home/jing/Phd_project/project_UCD_blca/blca_DATA/blca_DATA_bc3c_oct/gene_count.xls',
    sep='\t',   
    header=0,index_col=0
)
display(counts)

,D01,D02,D03,T01,T02,T03,S01,S02,S03,gene_name,gene_chr,gene_start,gene_end,gene_strand,gene_length,gene_biotype,gene_description,tf_family
gene_id,,,,,,,,,,,,,,,,,,
ENSG00000156508,236599,217166,230137,246465,277179,265535,291410,287477,279305,EEF1A1,6,73515750,73523797,-,5948,protein_coding,eukaryotic translation elongation factor 1 alp...,-
ENSG00000186081,173372,151121,162044,168813,186988,177195,144747,149534,158903,KRT5,12,52514575,52520687,-,4292,protein_coding,keratin 5 [Source:HGNC Symbol;Acc:HGNC:6442],-
ENSG00000080824,147634,114075,138470,120147,137207,134926,93481,97859,87311,HSP90AA1,14,102080738,102139699,-,5248,protein_coding,heat shock protein 90 alpha family class A mem...,-
ENSG00000187134,105703,95969,88389,103744,120824,114376,95104,108179,106387,AKR1C1,10,4963253,4983283,+,8765,protein_coding,aldo-keto reductase family 1 member C1 [Source...,-
ENSG00000196139,93872,97803,90777,109018,125059,116421,101868,96017,98182,AKR1C3,10,5035354,5107686,+,4532,protein_coding,aldo-keto reductase family 1 member C3 [Source...,-
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
ENSG00000275063,0,0,0,0,0,0,0,0,0,AC233755.1,KI270726.1,41444,41876,+,351,protein_coding,immunoglobulin heavy variable 4-38-2-like [Sou...,-
ENSG00000275405,0,0,0,0,0,0,0,0,0,RF00003,KI270713.1,21861,22024,-,164,snRNA,NaN,-
ENSG00000275987,0,0,0,0,0,0,0,0,0,RF00003,KI270713.1,30437,30580,-,144,snRNA,NaN,-


In [20]:
smad_df  =pd.read_csv(os.path.join(files_path,'augmented_Group_S_vs_D_DEGs.csv'),index_col=0)
display(smad_df)

,V1,V2,V3,V4,V5,V6,V7,V8,V9,V10,V11,V12,V13,V14,V15,V16,V17,V18,V19,V20
ENSG00000156508,0.267818,0.250431,0.258339,0.181589,0.250753,0.219754,0.259317,0.200100,0.236601,0.227299,0.330536,0.244935,0.227655,0.239768,0.164381,0.246356,0.214813,0.262664,0.294040,0.320645
ENSG00000186081,-0.270659,-0.214579,-0.177583,-0.122605,-0.238233,-0.140340,-0.112750,-0.102218,-0.114194,-0.177293,-0.259021,-0.224725,-0.077319,-0.205527,-0.099922,-0.103619,-0.103381,-0.179374,-0.202944,-0.251690
ENSG00000080824,-0.648882,-0.609840,-0.621077,-0.519491,-0.620831,-0.575570,-0.539669,-0.512242,-0.535387,-0.558118,-0.690562,-0.594507,-0.524116,-0.536778,-0.495778,-0.536145,-0.520466,-0.597496,-0.589405,-0.680019
ENSG00000075624,-0.553327,-0.447304,-0.530273,-0.409481,-0.429479,-0.424402,-0.449047,-0.321024,-0.462650,-0.421495,-0.581068,-0.452151,-0.449315,-0.367896,-0.355937,-0.351615,-0.408055,-0.472188,-0.447125,-0.475379
ENSG00000096384,-0.631544,-0.577096,-0.566570,-0.477981,-0.583133,-0.510339,-0.492897,-0.451967,-0.488453,-0.526770,-0.657332,-0.566211,-0.466472,-0.519741,-0.444723,-0.471725,-0.471840,-0.551671,-0.561901,-0.627111
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
ENSG00000053702,-2.002080,-1.689436,-1.561071,0.541366,-2.684528,-2.770834,-1.487364,-2.053640,-1.062125,-1.418561,-2.635171,-3.347576,-1.455604,-2.976164,-0.683174,-2.254330,0.179689,-3.332501,-2.159110,-5.245321
ENSG00000184860,-4.867906,-3.830393,-3.064666,-2.068944,-3.196792,-2.044355,-2.590525,-1.043911,-3.145863,-3.208080,-4.767025,-3.626019,-2.830786,-3.678875,-1.930937,-1.055293,-1.382268,-3.876558,-3.180868,-3.062467
ENSG00000279259,2.519366,3.534411,2.960847,1.804856,1.837106,1.713883,2.482235,2.595317,2.774206,2.618790,5.358544,1.328676,3.249234,0.935202,1.353173,1.983776,2.747092,3.341848,2.344522,3.813338
ENSG00000227359,-4.315378,-2.366259,-2.934172,-1.755826,-2.419720,-1.921446,-2.700961,-0.303094,-2.901534,-2.372890,-3.551570,-3.395520,-2.468915,-3.105285,-1.447017,-0.934478,-1.227321,-3.039423,-2.944456,-2.242576


In [21]:
smad_df['symbol']= counts.loc[smad_df.index,'gene_name']
display(smad_df)

,V1,V2,V3,V4,V5,V6,V7,V8,V9,V10,...,V12,V13,V14,V15,V16,V17,V18,V19,V20,symbol
ENSG00000156508,0.267818,0.250431,0.258339,0.181589,0.250753,0.219754,0.259317,0.200100,0.236601,0.227299,...,0.244935,0.227655,0.239768,0.164381,0.246356,0.214813,0.262664,0.294040,0.320645,EEF1A1
ENSG00000186081,-0.270659,-0.214579,-0.177583,-0.122605,-0.238233,-0.140340,-0.112750,-0.102218,-0.114194,-0.177293,...,-0.224725,-0.077319,-0.205527,-0.099922,-0.103619,-0.103381,-0.179374,-0.202944,-0.251690,KRT5
ENSG00000080824,-0.648882,-0.609840,-0.621077,-0.519491,-0.620831,-0.575570,-0.539669,-0.512242,-0.535387,-0.558118,...,-0.594507,-0.524116,-0.536778,-0.495778,-0.536145,-0.520466,-0.597496,-0.589405,-0.680019,HSP90AA1
ENSG00000075624,-0.553327,-0.447304,-0.530273,-0.409481,-0.429479,-0.424402,-0.449047,-0.321024,-0.462650,-0.421495,...,-0.452151,-0.449315,-0.367896,-0.355937,-0.351615,-0.408055,-0.472188,-0.447125,-0.475379,ACTB
ENSG00000096384,-0.631544,-0.577096,-0.566570,-0.477981,-0.583133,-0.510339,-0.492897,-0.451967,-0.488453,-0.526770,...,-0.566211,-0.466472,-0.519741,-0.444723,-0.471725,-0.471840,-0.551671,-0.561901,-0.627111,HSP90AB1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
ENSG00000053702,-2.002080,-1.689436,-1.561071,0.541366,-2.684528,-2.770834,-1.487364,-2.053640,-1.062125,-1.418561,...,-3.347576,-1.455604,-2.976164,-0.683174,-2.254330,0.179689,-3.332501,-2.159110,-5.245321,NRIP2
ENSG00000184860,-4.867906,-3.830393,-3.064666,-2.068944,-3.196792,-2.044355,-2.590525,-1.043911,-3.145863,-3.208080,...,-3.626019,-2.830786,-3.678875,-1.930937,-1.055293,-1.382268,-3.876558,-3.180868,-3.062467,SDR42E1
ENSG00000279259,2.519366,3.534411,2.960847,1.804856,1.837106,1.713883,2.482235,2.595317,2.774206,2.618790,...,1.328676,3.249234,0.935202,1.353173,1.983776,2.747092,3.341848,2.344522,3.813338,AC087741.3
ENSG00000227359,-4.315378,-2.366259,-2.934172,-1.755826,-2.419720,-1.921446,-2.700961,-0.303094,-2.901534,-2.372890,...,-3.395520,-2.468915,-3.105285,-1.447017,-0.934478,-1.227321,-3.039423,-2.944456,-2.242576,AC017074.1


In [22]:
smad_lfc = smad_df.set_index('symbol')
smad_lfc = smad_lfc.loc[smad_lfc.index.intersection(Data_norm_df.index)]
display(smad_lfc)

,V1,V2,V3,V4,V5,V6,V7,V8,V9,V10,V11,V12,V13,V14,V15,V16,V17,V18,V19,V20
GAPDH,-0.418152,-0.378634,-0.364679,-0.263892,-0.347446,-0.263073,-0.244382,-0.208759,-0.301027,-0.311857,-0.466722,-0.302955,-0.265909,-0.271792,-0.233447,-0.191365,-0.237988,-0.347047,-0.293119,-0.359380
HSPD1,-0.690897,-0.665134,-0.652550,-0.567147,-0.659815,-0.609033,-0.578545,-0.549631,-0.578327,-0.605370,-0.730267,-0.629661,-0.571679,-0.580900,-0.545108,-0.570301,-0.557540,-0.639824,-0.625826,-0.701300
HSPA8,-0.596020,-0.483657,-0.553548,-0.509347,-0.450671,-0.483620,-0.561689,-0.403153,-0.524346,-0.493066,-0.583500,-0.530125,-0.539842,-0.444983,-0.441469,-0.447797,-0.517435,-0.506071,-0.536500,-0.483699
SPP1,0.183543,0.160707,0.167465,0.152414,0.141925,0.156691,0.175856,0.061337,0.119418,0.116680,0.170852,0.137803,0.149321,0.075414,0.111390,0.155068,0.129231,0.120261,0.168496,0.121010
TXNRD1,-0.369827,-0.405291,-0.334293,-0.257711,-0.399601,-0.312548,-0.209274,-0.234407,-0.237951,-0.291079,-0.404563,-0.292291,-0.228591,-0.259964,-0.261069,-0.255030,-0.199669,-0.325166,-0.274368,-0.379084
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
CDC42,-0.954408,-1.133006,-1.025483,-0.750016,-1.203452,-0.308864,-0.369969,-0.566596,-0.732181,-0.832729,-1.402006,-0.358399,-0.148742,-0.727734,-0.468275,-0.550377,-0.827254,-0.567495,-0.956274,-0.945379
APOE,1.757115,0.958865,1.180892,0.361918,1.470654,1.152320,0.944884,0.782349,0.704361,1.015152,1.705620,1.908782,0.546101,1.553845,0.355148,0.860957,0.546481,1.397371,1.521237,2.344394
SATB1,1.838739,1.658157,1.726674,0.842176,1.095462,0.900912,0.918231,0.684494,1.300680,1.206626,2.673180,0.971805,1.317559,0.475854,0.570397,0.362617,0.930993,1.660736,0.936688,1.743425
SPDEF,-2.192339,-2.477598,-1.448971,-0.849881,-1.784082,-1.055109,-1.117169,-1.309503,-1.334458,-1.823078,-3.150591,-1.644710,-1.430505,-1.641612,-0.846442,-0.860945,-0.920115,-2.251125,-1.632585,-2.591862


In [23]:
smad_lfc.columns = ['SMAD_' + col for col in smad_lfc.columns]
display(smad_lfc)

,SMAD_V1,SMAD_V2,SMAD_V3,SMAD_V4,SMAD_V5,SMAD_V6,SMAD_V7,SMAD_V8,SMAD_V9,SMAD_V10,SMAD_V11,SMAD_V12,SMAD_V13,SMAD_V14,SMAD_V15,SMAD_V16,SMAD_V17,SMAD_V18,SMAD_V19,SMAD_V20
GAPDH,-0.418152,-0.378634,-0.364679,-0.263892,-0.347446,-0.263073,-0.244382,-0.208759,-0.301027,-0.311857,-0.466722,-0.302955,-0.265909,-0.271792,-0.233447,-0.191365,-0.237988,-0.347047,-0.293119,-0.359380
HSPD1,-0.690897,-0.665134,-0.652550,-0.567147,-0.659815,-0.609033,-0.578545,-0.549631,-0.578327,-0.605370,-0.730267,-0.629661,-0.571679,-0.580900,-0.545108,-0.570301,-0.557540,-0.639824,-0.625826,-0.701300
HSPA8,-0.596020,-0.483657,-0.553548,-0.509347,-0.450671,-0.483620,-0.561689,-0.403153,-0.524346,-0.493066,-0.583500,-0.530125,-0.539842,-0.444983,-0.441469,-0.447797,-0.517435,-0.506071,-0.536500,-0.483699
SPP1,0.183543,0.160707,0.167465,0.152414,0.141925,0.156691,0.175856,0.061337,0.119418,0.116680,0.170852,0.137803,0.149321,0.075414,0.111390,0.155068,0.129231,0.120261,0.168496,0.121010
TXNRD1,-0.369827,-0.405291,-0.334293,-0.257711,-0.399601,-0.312548,-0.209274,-0.234407,-0.237951,-0.291079,-0.404563,-0.292291,-0.228591,-0.259964,-0.261069,-0.255030,-0.199669,-0.325166,-0.274368,-0.379084
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
CDC42,-0.954408,-1.133006,-1.025483,-0.750016,-1.203452,-0.308864,-0.369969,-0.566596,-0.732181,-0.832729,-1.402006,-0.358399,-0.148742,-0.727734,-0.468275,-0.550377,-0.827254,-0.567495,-0.956274,-0.945379
APOE,1.757115,0.958865,1.180892,0.361918,1.470654,1.152320,0.944884,0.782349,0.704361,1.015152,1.705620,1.908782,0.546101,1.553845,0.355148,0.860957,0.546481,1.397371,1.521237,2.344394
SATB1,1.838739,1.658157,1.726674,0.842176,1.095462,0.900912,0.918231,0.684494,1.300680,1.206626,2.673180,0.971805,1.317559,0.475854,0.570397,0.362617,0.930993,1.660736,0.936688,1.743425
SPDEF,-2.192339,-2.477598,-1.448971,-0.849881,-1.784082,-1.055109,-1.117169,-1.309503,-1.334458,-1.823078,-3.150591,-1.644710,-1.430505,-1.641612,-0.846442,-0.860945,-0.920115,-2.251125,-1.632585,-2.591862


In [24]:
Data_norm_df[smad_lfc.columns] = 0

In [25]:
common_idx = Data_norm_df.index.intersection(smad_lfc.index)
Data_norm_df.loc[common_idx, smad_lfc.columns] = smad_lfc.loc[common_idx, smad_lfc.columns]

/tmp/ipykernel_1458237/1508837703.py:2: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[ 0.80006032 -0.38077041  0.6067599   0.14613308  1.04119834  0.64850065
  0.33453216 -0.3657465  -0.14819256 -0.93501364  0.2000188  -0.20626723
 -0.54663673  0.37336362 -0.33119211 -0.45995065 -0.39305895  0.17315114
  1.80734362  1.34024181  0.32793335  0.54169673  0.40003945  1.75711503
  0.2348163   0.40610619 -0.27040384  0.74169684 -0.37149541 -0.22467869
 -0.89065041 -0.14976776 -0.35416148 -0.35196214 -0.30704198 -1.10837272
 -0.68850645 -0.18754113 -0.25961763  0.44222036  0.71956938  0.31011175
 -0.87380603 -0.25761065 -0.35763975  0.29971468  0.49584234  0.71244868
 -0.12837556 -0.82204431 -0.56970768  0.51936367 -0.25994771 -0.31303677
  0.29745721  0.52000627 -0.43751303  0.81274654 -0.30501018 -0.4858665
 -1.15673013  0.61185731 -0.83667314 -1.04763391 -0.8749282  -0.85027437
 -0.80667006 -1.15082152 -0.5

In [26]:
Data_norm_df

,ASG002_BC3C_24H:A10,ASG002_BC3C_24H:A11,ASG002_BC3C_24H:A19,ASG002_BC3C_24H:A20,ASG002_BC3C_24H:A21,ASG002_BC3C_24H:B10,ASG002_BC3C_24H:B11,ASG002_BC3C_24H:B14,ASG002_BC3C_24H:B15,ASG002_BC3C_24H:C13,...,SMAD_V11,SMAD_V12,SMAD_V13,SMAD_V14,SMAD_V15,SMAD_V16,SMAD_V17,SMAD_V18,SMAD_V19,SMAD_V20
AARS,-0.496854,0.288446,0.189747,-0.016454,0.080746,0.282346,0.326246,0.303046,0.387546,0.434746,...,0.846946,0.739186,0.728520,0.661815,0.627668,0.680916,0.700434,0.746670,0.755803,0.789433
ABCB6,-0.658596,-0.142196,-0.075397,-0.383796,-0.199996,-0.074197,0.108804,-0.399196,-0.227496,-0.041146,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
ABCC5,-0.080204,0.231996,-0.329354,-0.225204,0.278446,0.034696,0.396596,-0.255704,-0.254154,-0.002004,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
ABCF1,0.202535,0.602335,0.403335,0.313134,-0.083265,-0.056365,-0.387216,0.097235,0.540634,-0.262666,...,-0.374807,-0.352552,-0.198751,-0.306206,-0.173180,-0.224540,-0.200641,-0.301348,-0.327189,-0.400115
ABCF3,-0.520919,-0.192819,0.001032,-0.096419,0.210881,-0.731118,0.095381,0.313481,-0.078018,-0.257669,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
ZNF395,1.796254,1.773154,0.668354,0.825354,0.350654,0.205954,-0.998346,0.235354,0.230653,0.440854,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
ZNF451,-0.244519,0.116732,0.058081,-0.178169,-0.080619,-0.139119,0.064681,-0.083019,-0.272119,-0.164119,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
ZNF586,0.097627,0.061027,-0.337573,0.098427,-0.338173,0.017627,-0.197423,-0.078473,-0.300473,-0.147773,...,-0.796426,-0.374528,-0.369905,-0.345740,-0.201382,-0.056259,-0.226123,-0.499067,-0.395016,-0.387449
ZNF589,0.608573,0.106123,-0.014126,-0.003677,-0.123477,-0.243377,0.044623,-0.021477,0.201823,-0.413227,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000


In [27]:
genes = Data_norm_df.index.tolist()
print('Landmark genes list: ', len(genes), genes)

n_genes = len(genes)

Landmark genes list:  978 ['AARS', 'ABCB6', 'ABCC5', 'ABCF1', 'ABCF3', 'ABHD4', 'ABHD6', 'ABL1', 'ACAA1', 'ACAT2', 'ACBD3', 'ACD', 'ACLY', 'ACOT9', 'ADAM10', 'ADAT1', 'ADGRE5', 'ADGRG1', 'ADH5', 'ADI1', 'ADO', 'ADRB2', 'AGL', 'AKAP8', 'AKAP8L', 'AKR7A2', 'AKT1', 'ALAS1', 'ALDH7A1', 'ALDOA', 'ALDOC', 'AMDHD2', 'ANKRD10', 'ANO10', 'ANXA7', 'APBB2', 'APOE', 'APP', 'APPBP2', 'ARFIP2', 'ARHGAP1', 'ARHGEF12', 'ARHGEF2', 'ARID4B', 'ARID5B', 'ARL4C', 'ARNT2', 'ARPP19', 'ASAH1', 'ASCC3', 'ATF1', 'ATF5', 'ATF6', 'ATG3', 'ATMIN', 'ATP11B', 'ATP1B1', 'ATP2C1', 'ATP6V0B', 'ATP6V1D', 'AURKA', 'AURKB', 'AXIN1', 'B4GAT1', 'BACE2', 'BAD', 'BAG3', 'BAMBI', 'BAX', 'BCL2', 'BCL7B', 'BDH1', 'BECN1', 'BHLHE40', 'BID', 'BIRC2', 'BIRC5', 'BLCAP', 'BLMH', 'BLVRA', 'BMP4', 'BNIP3', 'BNIP3L', 'BPHL', 'BRCA1', 'BTK', 'BUB1B', 'BZW2', 'C2CD2', 'C2CD2L', 'C2CD5', 'C5', 'CAB39', 'CALM3', 'CALU', 'CAMSAP2', 'CANT1', 'CAPN1', 'CARMIL1', 'CASC3', 'CASK', 'CASP10', 'CASP2', 'CASP3', 'CASP7', 'CAST', 'CAT', 'CBLB', 'CBR1

## Inhibitor concentrations, IC50, and perturbation matrices

In [28]:
inhib_conc_matrix = np.zeros((n_modules, n_experiments))
ic50_matrix = np.ones((n_modules, n_experiments))
gamma_matrix = np.zeros((n_modules, n_experiments))

In [29]:

for i, module in enumerate(modules):
    drugs_for_module = IC50_df.Drug[IC50_df.Module == module].tolist()
    for drug in drugs_for_module:
        # get IC50 for this drug
        ic50 = IC50_df.IC50[IC50_df.Drug == drug].values
#       gamma = IC50_df.Gamma[IC50_df.Drug == drug].values
        print(drug, ic50)
        assert ic50.size == 1
#       assert gamma.size == 1
        # get experiments with this drug
        exp_with_drug = sig_info_df.index[sig_info_df.pert_drug == drug].tolist()
        print(exp_with_drug) 
        for exp_id in exp_with_drug:
            j = exp_ids.index(exp_id)
            print(j)
            # extract inhibitor concentration
            inhib_conc = sig_info_df.dose_float[sig_info_df.index == exp_id].values
            assert inhib_conc.size == 1
            # insert values in matrices
            inhib_conc_matrix[i, j] = inhib_conc.item()
            ic50_matrix[i, j] = ic50.item()
#           gamma_matrix[i, j] = gamma.item()


JNJ-7706621 [0.027]
['ASG002_BC3C_24H:N13', 'ASG002_BC3C_24H:N14', 'ASG002_BC3C_24H:N15']
51
52
53
PHA-793887 [0.18]
['ASG002_BC3C_24H:L01', 'ASG002_BC3C_24H:L02', 'ASG002_BC3C_24H:L03']
43
44
45
roscovitine [2.]
['ASG002_BC3C_24H:E22', 'ASG002_BC3C_24H:E23', 'ASG002_BC3C_24H:E24']
15
16
17
alvocidib [0.12]
['ASG002_BC3C_24H:F05', 'ASG002_BC3C_24H:F06']
21
22
palbociclib [0.045]
['ASG002_BC3C_24H:P16', 'ASG002_BC3C_24H:P18']
66
67
afatinib [0.03]
['ASG002_BC3C_24H:N23', 'ASG002_BC3C_24H:N24']
56
57
erlotinib [0.006]
['ASG002_BC3C_24H:H16', 'ASG002_BC3C_24H:H17', 'ASG002_BC3C_24H:H18']
30
31
32
gefitinib [0.171]
['ASG002_BC3C_24H:F16', 'ASG002_BC3C_24H:F17', 'ASG002_BC3C_24H:F18']
25
26
27
lapatinib [0.0324]
['ASG002_BC3C_24H:I13', 'ASG002_BC3C_24H:I14', 'ASG002_BC3C_24H:I15']
38
39
40
vandetanib [1.5]
['ASG002_BC3C_24H:N20', 'ASG002_BC3C_24H:N21']
54
55
dienestrol [200.]
['ASG002_BC3C_24H:M22', 'ASG002_BC3C_24H:M23', 'ASG002_BC3C_24H:M24']
48
49
50
estradiol-cypionate [200.]
['ASG002_B

In [30]:
inhib_conc_matrix

array([[ 0.  ,  0.  ,  0.  ,  0.  ,  0.  ,  0.  ,  0.  ,  0.  ,  0.  ,
         0.  ,  0.  ,  0.  ,  0.  ,  0.  ,  0.  , 10.  ,  1.11,  0.12,
         0.  ,  0.  ,  0.  ,  0.  ,  0.  ,  0.  ,  0.  ,  0.  ,  0.  ,
         0.  ,  0.  ,  0.  ,  0.  ,  0.  ,  0.  ,  0.  ,  0.  ,  0.  ,
         0.  ,  0.  ,  0.  ,  0.  ,  0.  ,  0.  ,  0.  , 10.  ,  1.11,
         0.12,  0.  ,  0.  ,  0.  ,  0.  ,  0.  , 10.  ,  1.11,  0.12,
         0.  ,  0.  ,  0.  ,  0.  ,  0.  ,  0.  ,  0.  ,  0.  ,  0.  ,
         0.  ,  0.  ,  0.  ,  0.  ,  0.  ,  0.  ,  0.  ,  0.  ,  0.  ,
         0.  ,  0.  ,  0.  ,  0.  ,  0.  ,  0.  ,  0.  ,  0.  ,  0.  ,
         0.  ,  0.  ,  0.  ,  0.  ,  0.  ],
       [ 0.  ,  0.  ,  0.  ,  0.  ,  0.  ,  0.  ,  0.  ,  0.  ,  0.  ,
         0.  ,  0.  ,  0.  ,  0.  ,  0.  ,  0.  ,  0.  ,  0.  ,  0.  ,
         0.  ,  0.  ,  0.  ,  1.11,  0.12,  0.  ,  0.  ,  0.  ,  0.  ,
         0.  ,  0.  ,  0.  ,  0.  ,  0.  ,  0.  ,  0.  ,  0.  ,  0.  ,
         0.  ,  0.  ,  0.  ,  0. 

In [31]:
# transform matrices into pandas dfs for export with row and column names
inhib_conc_df = pd.DataFrame(inhib_conc_matrix, index = modules, columns = exp_ids)
ic50_df = pd.DataFrame(ic50_matrix, index = modules, columns = exp_ids)
# gamma_df = pd.DataFrame(gamma_matrix, index = modules, columns = exp_ids)

# create binary perturbation matrix
pert_df = pd.DataFrame(
    np.where(inhib_conc_matrix != 0, 1, 0),
    index = inhib_conc_df.index,
    columns = inhib_conc_df.columns,
)

In [32]:
display(ic50_df)
# display(gamma_df)
display(inhib_conc_df)
display(pert_df)

,ASG002_BC3C_24H:A10,ASG002_BC3C_24H:A11,ASG002_BC3C_24H:A19,ASG002_BC3C_24H:A20,ASG002_BC3C_24H:A21,ASG002_BC3C_24H:B10,ASG002_BC3C_24H:B11,ASG002_BC3C_24H:B14,ASG002_BC3C_24H:B15,ASG002_BC3C_24H:C13,...,MOAR010_BC3C_24H:A15,MOAR010_BC3C_24H:J22,MOAR010_BC3C_24H:J23,MOAR010_BC3C_24H:J24,MOAR010_BC3C_24H:K07,MOAR010_BC3C_24H:K08,MOAR010_BC3C_24H:K09,MOAR011_BC3C_24H:F07,MOAR011_BC3C_24H:F08,MOAR011_BC3C_24H:F09
CDK1_2,1.00000,1.00000,1.0000,1.0000,1.0000,1.0000,1.0000,1.00,1.00,1.00,...,1.00,1.00,1.00,1.00,1.0000,1.0000,1.0000,1.00,1.00,1.00
CDK4_6,1.00000,1.00000,1.0000,1.0000,1.0000,1.0000,1.0000,1.00,1.00,1.00,...,1.00,1.00,1.00,1.00,1.0000,1.0000,1.0000,1.00,1.00,1.00
EGFR,1.00000,1.00000,1.0000,1.0000,1.0000,1.0000,1.0000,1.00,1.00,1.00,...,1.00,1.00,1.00,1.00,1.0000,1.0000,1.0000,1.00,1.00,1.00
Estrogen,1.00000,1.00000,1.0000,1.0000,1.0000,1.0000,1.0000,1.00,1.00,1.00,...,1.00,1.00,1.00,1.00,1.0000,1.0000,1.0000,1.00,1.00,1.00
FGFR,1.00000,1.00000,1.0000,1.0000,1.0000,1.0000,1.0000,1.00,1.00,1.00,...,1.00,1.00,1.00,1.00,1.0000,1.0000,1.0000,1.00,1.00,1.00
PI3K,0.00262,0.00262,0.1595,0.1595,0.1595,1.0000,1.0000,1.00,1.00,1.00,...,1.00,1.00,1.00,1.00,1.0000,1.0000,1.0000,1.00,1.00,1.00
p53,1.00000,1.00000,1.0000,1.0000,1.0000,0.0018,0.0018,9.75,9.75,0.54,...,9.75,62.15,62.15,62.15,0.0264,0.0264,0.0264,62.15,62.15,62.15
TOP2A,1.00000,1.00000,1.0000,1.0000,1.0000,1.0000,1.0000,1.00,1.00,1.00,...,1.00,1.00,1.00,1.00,1.0000,1.0000,1.0000,1.00,1.00,1.00
Src,1.00000,1.00000,1.0000,1.0000,1.0000,1.0000,1.0000,1.00,1.00,1.00,...,1.00,1.00,1.00,1.00,1.0000,1.0000,1.0000,1.00,1.00,1.00
SMAD3,1.00000,1.00000,1.0000,1.0000,1.0000,1.0000,1.0000,1.00,1.00,1.00,...,1.00,1.00,1.00,1.00,1.0000,1.0000,1.0000,1.00,1.00,1.00


,ASG002_BC3C_24H:A10,ASG002_BC3C_24H:A11,ASG002_BC3C_24H:A19,ASG002_BC3C_24H:A20,ASG002_BC3C_24H:A21,ASG002_BC3C_24H:B10,ASG002_BC3C_24H:B11,ASG002_BC3C_24H:B14,ASG002_BC3C_24H:B15,ASG002_BC3C_24H:C13,...,MOAR010_BC3C_24H:A15,MOAR010_BC3C_24H:J22,MOAR010_BC3C_24H:J23,MOAR010_BC3C_24H:J24,MOAR010_BC3C_24H:K07,MOAR010_BC3C_24H:K08,MOAR010_BC3C_24H:K09,MOAR011_BC3C_24H:F07,MOAR011_BC3C_24H:F08,MOAR011_BC3C_24H:F09
CDK1_2,0.0,0.00,0.0,0.00,0.00,0.0,0.00,0.00,0.00,0.0,...,0.00,0.0,0.00,0.00,0.0,0.00,0.00,0.0,0.00,0.00
CDK4_6,0.0,0.00,0.0,0.00,0.00,0.0,0.00,0.00,0.00,0.0,...,0.00,0.0,0.00,0.00,0.0,0.00,0.00,0.0,0.00,0.00
EGFR,0.0,0.00,0.0,0.00,0.00,0.0,0.00,0.00,0.00,0.0,...,0.00,0.0,0.00,0.00,0.0,0.00,0.00,0.0,0.00,0.00
Estrogen,0.0,0.00,0.0,0.00,0.00,0.0,0.00,0.00,0.00,0.0,...,0.00,0.0,0.00,0.00,0.0,0.00,0.00,0.0,0.00,0.00
FGFR,0.0,0.00,0.0,0.00,0.00,0.0,0.00,0.00,0.00,0.0,...,0.00,0.0,0.00,0.00,0.0,0.00,0.00,0.0,0.00,0.00
PI3K,10.0,1.11,10.0,1.11,0.12,0.0,0.00,0.00,0.00,0.0,...,0.00,0.0,0.00,0.00,0.0,0.00,0.00,0.0,0.00,0.00
p53,0.0,0.00,0.0,0.00,0.00,10.0,1.11,1.11,0.08,10.0,...,1.11,10.0,3.33,1.11,10.0,3.33,1.11,10.0,3.33,1.11
TOP2A,0.0,0.00,0.0,0.00,0.00,0.0,0.00,0.00,0.00,0.0,...,0.00,0.0,0.00,0.00,0.0,0.00,0.00,0.0,0.00,0.00
Src,0.0,0.00,0.0,0.00,0.00,0.0,0.00,0.00,0.00,0.0,...,0.00,0.0,0.00,0.00,0.0,0.00,0.00,0.0,0.00,0.00
SMAD3,0.0,0.00,0.0,0.00,0.00,0.0,0.00,0.00,0.00,0.0,...,0.00,0.0,0.00,0.00,0.0,0.00,0.00,0.0,0.00,0.00


,ASG002_BC3C_24H:A10,ASG002_BC3C_24H:A11,ASG002_BC3C_24H:A19,ASG002_BC3C_24H:A20,ASG002_BC3C_24H:A21,ASG002_BC3C_24H:B10,ASG002_BC3C_24H:B11,ASG002_BC3C_24H:B14,ASG002_BC3C_24H:B15,ASG002_BC3C_24H:C13,...,MOAR010_BC3C_24H:A15,MOAR010_BC3C_24H:J22,MOAR010_BC3C_24H:J23,MOAR010_BC3C_24H:J24,MOAR010_BC3C_24H:K07,MOAR010_BC3C_24H:K08,MOAR010_BC3C_24H:K09,MOAR011_BC3C_24H:F07,MOAR011_BC3C_24H:F08,MOAR011_BC3C_24H:F09
CDK1_2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
CDK4_6,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
EGFR,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
Estrogen,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
FGFR,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
PI3K,1,1,1,1,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
p53,0,0,0,0,0,1,1,1,1,1,...,1,1,1,1,1,1,1,1,1,1
TOP2A,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
Src,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
SMAD3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [33]:
cols = list(smad_lfc.columns)

ic50_df[cols] = 1.0
inhib_conc_df[cols] = 0.0
pert_df[cols] = 0

In [34]:
ic50_df

,ASG002_BC3C_24H:A10,ASG002_BC3C_24H:A11,ASG002_BC3C_24H:A19,ASG002_BC3C_24H:A20,ASG002_BC3C_24H:A21,ASG002_BC3C_24H:B10,ASG002_BC3C_24H:B11,ASG002_BC3C_24H:B14,ASG002_BC3C_24H:B15,ASG002_BC3C_24H:C13,...,SMAD_V11,SMAD_V12,SMAD_V13,SMAD_V14,SMAD_V15,SMAD_V16,SMAD_V17,SMAD_V18,SMAD_V19,SMAD_V20
CDK1_2,1.00000,1.00000,1.0000,1.0000,1.0000,1.0000,1.0000,1.00,1.00,1.00,...,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
CDK4_6,1.00000,1.00000,1.0000,1.0000,1.0000,1.0000,1.0000,1.00,1.00,1.00,...,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
EGFR,1.00000,1.00000,1.0000,1.0000,1.0000,1.0000,1.0000,1.00,1.00,1.00,...,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
Estrogen,1.00000,1.00000,1.0000,1.0000,1.0000,1.0000,1.0000,1.00,1.00,1.00,...,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
FGFR,1.00000,1.00000,1.0000,1.0000,1.0000,1.0000,1.0000,1.00,1.00,1.00,...,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
PI3K,0.00262,0.00262,0.1595,0.1595,0.1595,1.0000,1.0000,1.00,1.00,1.00,...,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
p53,1.00000,1.00000,1.0000,1.0000,1.0000,0.0018,0.0018,9.75,9.75,0.54,...,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
TOP2A,1.00000,1.00000,1.0000,1.0000,1.0000,1.0000,1.0000,1.00,1.00,1.00,...,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
Src,1.00000,1.00000,1.0000,1.0000,1.0000,1.0000,1.0000,1.00,1.00,1.00,...,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
SMAD3,1.00000,1.00000,1.0000,1.0000,1.0000,1.0000,1.0000,1.00,1.00,1.00,...,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0


In [71]:
ic50_df.loc['SMAD3',smad_lfc.columns] = 7

In [72]:
inhib_conc_df.loc['SMAD3',smad_lfc.columns] = 10

In [73]:
pert_df.loc['SMAD3',smad_lfc.columns] = 1

In [74]:
display(ic50_df)
# display(gamma_df)
display(inhib_conc_df)
display(pert_df)

,ASG002_BC3C_24H:A10,ASG002_BC3C_24H:A11,ASG002_BC3C_24H:A19,ASG002_BC3C_24H:A20,ASG002_BC3C_24H:A21,ASG002_BC3C_24H:B10,ASG002_BC3C_24H:B11,ASG002_BC3C_24H:B14,ASG002_BC3C_24H:B15,ASG002_BC3C_24H:C13,...,SMAD_V11,SMAD_V12,SMAD_V13,SMAD_V14,SMAD_V15,SMAD_V16,SMAD_V17,SMAD_V18,SMAD_V19,SMAD_V20
CDK1_2,1.00000,1.00000,1.0000,1.0000,1.0000,1.0000,1.0000,1.00,1.00,1.00,...,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
CDK4_6,1.00000,1.00000,1.0000,1.0000,1.0000,1.0000,1.0000,1.00,1.00,1.00,...,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
EGFR,1.00000,1.00000,1.0000,1.0000,1.0000,1.0000,1.0000,1.00,1.00,1.00,...,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
Estrogen,1.00000,1.00000,1.0000,1.0000,1.0000,1.0000,1.0000,1.00,1.00,1.00,...,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
FGFR,1.00000,1.00000,1.0000,1.0000,1.0000,1.0000,1.0000,1.00,1.00,1.00,...,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
PI3K,0.00262,0.00262,0.1595,0.1595,0.1595,1.0000,1.0000,1.00,1.00,1.00,...,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
p53,1.00000,1.00000,1.0000,1.0000,1.0000,0.0018,0.0018,9.75,9.75,0.54,...,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
TOP2A,1.00000,1.00000,1.0000,1.0000,1.0000,1.0000,1.0000,1.00,1.00,1.00,...,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
Src,1.00000,1.00000,1.0000,1.0000,1.0000,1.0000,1.0000,1.00,1.00,1.00,...,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
SMAD3,1.00000,1.00000,1.0000,1.0000,1.0000,1.0000,1.0000,1.00,1.00,1.00,...,7.0,7.0,7.0,7.0,7.0,7.0,7.0,7.0,7.0,7.0


,ASG002_BC3C_24H:A10,ASG002_BC3C_24H:A11,ASG002_BC3C_24H:A19,ASG002_BC3C_24H:A20,ASG002_BC3C_24H:A21,ASG002_BC3C_24H:B10,ASG002_BC3C_24H:B11,ASG002_BC3C_24H:B14,ASG002_BC3C_24H:B15,ASG002_BC3C_24H:C13,...,SMAD_V11,SMAD_V12,SMAD_V13,SMAD_V14,SMAD_V15,SMAD_V16,SMAD_V17,SMAD_V18,SMAD_V19,SMAD_V20
CDK1_2,0.0,0.00,0.0,0.00,0.00,0.0,0.00,0.00,0.00,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
CDK4_6,0.0,0.00,0.0,0.00,0.00,0.0,0.00,0.00,0.00,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
EGFR,0.0,0.00,0.0,0.00,0.00,0.0,0.00,0.00,0.00,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Estrogen,0.0,0.00,0.0,0.00,0.00,0.0,0.00,0.00,0.00,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
FGFR,0.0,0.00,0.0,0.00,0.00,0.0,0.00,0.00,0.00,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
PI3K,10.0,1.11,10.0,1.11,0.12,0.0,0.00,0.00,0.00,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
p53,0.0,0.00,0.0,0.00,0.00,10.0,1.11,1.11,0.08,10.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
TOP2A,0.0,0.00,0.0,0.00,0.00,0.0,0.00,0.00,0.00,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Src,0.0,0.00,0.0,0.00,0.00,0.0,0.00,0.00,0.00,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
SMAD3,0.0,0.00,0.0,0.00,0.00,0.0,0.00,0.00,0.00,0.0,...,10.0,10.0,10.0,10.0,10.0,10.0,10.0,10.0,10.0,10.0


,ASG002_BC3C_24H:A10,ASG002_BC3C_24H:A11,ASG002_BC3C_24H:A19,ASG002_BC3C_24H:A20,ASG002_BC3C_24H:A21,ASG002_BC3C_24H:B10,ASG002_BC3C_24H:B11,ASG002_BC3C_24H:B14,ASG002_BC3C_24H:B15,ASG002_BC3C_24H:C13,...,SMAD_V11,SMAD_V12,SMAD_V13,SMAD_V14,SMAD_V15,SMAD_V16,SMAD_V17,SMAD_V18,SMAD_V19,SMAD_V20
CDK1_2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
CDK4_6,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
EGFR,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
Estrogen,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
FGFR,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
PI3K,1,1,1,1,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
p53,0,0,0,0,0,1,1,1,1,1,...,0,0,0,0,0,0,0,0,0,0
TOP2A,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
Src,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
SMAD3,0,0,0,0,0,0,0,0,0,0,...,1,1,1,1,1,1,1,1,1,1


## Global responses for DPD modules

According to our discussion, $R$ for DPD vectors can not be calculated with the same formula as for pathway activities. Instead we are assuming:

\begin{equation}
R_{DPD},j = DPD = STV_{DPD} \cdot Data_j
\end{equation}

In [75]:
# load STV data frame
STVs = pd.read_excel(os.path.join(info_dir, "ALL_DATA_2020_post.xlsx"), sheet_name = "STV", index_col = 0)
STV_df = pd.DataFrame(np.zeros((len(Data_norm_df.index), 7)), index = Data_norm_df.index, columns = STVs.columns)
STV_df.loc[STVs.index] = STVs

display(STV_df)

,DPD_BL,DPD_onc,DPD_risk,DPD_T2,DPD_Ta,DPD_immuno,DPD_immuno_0
AARS,0.0,0.078480,0.000000,0.0,0.0,0.0,0.000000
ABCB6,0.0,0.027123,-0.153918,0.0,0.0,0.0,0.016004
ABCC5,0.0,0.002120,0.000000,0.0,0.0,0.0,-0.006153
ABCF1,0.0,-0.028956,-0.026418,0.0,0.0,0.0,-0.036127
ABCF3,0.0,0.036668,0.000000,0.0,0.0,0.0,-0.004999
...,...,...,...,...,...,...,...
ZNF395,0.0,0.000000,0.000000,0.0,0.0,0.0,0.010048
ZNF451,0.0,0.013401,0.000000,0.0,0.0,0.0,-0.009844
ZNF586,0.0,0.000000,0.006341,0.0,0.0,0.0,0.039816
ZNF589,0.0,0.000000,0.000000,0.0,0.0,0.0,0.018821


In [76]:
STV_df = STV_df[['DPD_onc','DPD_risk','DPD_immuno_0']]

In [77]:
# create empty DPD data frame
DPD_df = pd.DataFrame(
    np.zeros((len(Data_norm_df.columns), len(STV_df.columns))),
    index = Data_norm_df.columns,
    columns = STV_df.columns,
)

# populate
for exp_id in DPD_df.index:
    for state in STV_df.columns:
        DPD_df.loc[exp_id, state] = np.dot(Data_norm_df.T.loc[exp_id], STV_df.loc[:, state])

display(DPD_df)

,DPD_onc,DPD_risk,DPD_immuno_0
ASG002_BC3C_24H:A10,-0.610595,-0.925192,-0.355927
ASG002_BC3C_24H:A11,-0.935965,-0.306921,-0.645334
ASG002_BC3C_24H:A19,-0.048492,0.525629,-0.050693
ASG002_BC3C_24H:A20,-0.647615,-0.031134,-0.482723
ASG002_BC3C_24H:A21,0.031647,-0.330139,0.330559
...,...,...,...
SMAD_V16,-0.604095,-0.674103,-0.376065
SMAD_V17,-0.604703,-0.921762,-0.344518
SMAD_V18,-0.868657,-1.018414,-0.435230
SMAD_V19,-0.735838,-0.894276,-0.458314


In [78]:
# transform to R global
R_global_DPD_df = DPD_df.T
display(R_global_DPD_df)

,ASG002_BC3C_24H:A10,ASG002_BC3C_24H:A11,ASG002_BC3C_24H:A19,ASG002_BC3C_24H:A20,ASG002_BC3C_24H:A21,ASG002_BC3C_24H:B10,ASG002_BC3C_24H:B11,ASG002_BC3C_24H:B14,ASG002_BC3C_24H:B15,ASG002_BC3C_24H:C13,...,SMAD_V11,SMAD_V12,SMAD_V13,SMAD_V14,SMAD_V15,SMAD_V16,SMAD_V17,SMAD_V18,SMAD_V19,SMAD_V20
DPD_onc,-0.610595,-0.935965,-0.048492,-0.647615,0.031647,0.185961,0.911044,-0.079380,-0.459700,-0.521396,...,-1.136556,-0.816184,-0.719059,-0.512580,-0.501753,-0.604095,-0.604703,-0.868657,-0.735838,-1.067839
DPD_risk,-0.925192,-0.306921,0.525629,-0.031134,-0.330139,0.097778,-0.162965,-0.190680,0.055438,-0.322754,...,-1.388249,-0.919212,-0.884448,-0.677994,-0.688052,-0.674103,-0.921762,-1.018414,-0.894276,-1.174936
DPD_immuno_0,-0.355927,-0.645334,-0.050693,-0.482723,0.330559,-0.716308,0.157080,-0.374256,-0.223585,-0.426159,...,-0.481136,-0.512503,-0.448582,-0.323753,-0.239786,-0.376065,-0.344518,-0.435230,-0.458314,-0.516428


## Save outputs

In [79]:
# save metadata as pickle
all_metadata = {
    "modules": modules,
    "n_modules": n_modules,
    "drugs": drugs,
    "n_drugs": n_drugs,
    "exp_ids": exp_ids,
    "n_experiments": n_experiments,
    "genes": genes,
    "n_genes": n_genes,
}

print(all_metadata)

with open(os.path.join(out_dir, "metadata.pickle"), "wb") as f:
    pickle.dump(all_metadata, f, protocol = pickle.HIGHEST_PROTOCOL)

{'modules': ['CDK1_2', 'CDK4_6', 'EGFR', 'Estrogen', 'FGFR', 'PI3K', 'p53', 'TOP2A', 'Src', 'SMAD3'], 'n_modules': 10, 'drugs': ['dasatinib', 'RITA', 'daunorubicin', 'afatinib', 'serdemetan', 'vandetanib', 'AS-605240', 'SAR405838', 'taselisib', 'estradiol-cypionate', 'epirubicin', 'palbociclib', 'HLI-373', 'alvocidib', 'dienestrol', 'sorafenib', 'roscovitine', 'gefitinib', 'nutlin-3', 'masitinib', 'PI-103', 'ponatinib', 'idarubicin', 'erlotinib', 'JNJ-7706621', 'mitoxantrone', 'lapatinib', 'LY-294002', 'Agent2', 'raloxifene', 'AMG-232', 'AZD-8055', 'PHA-793887'], 'n_drugs': 33, 'exp_ids': ['ASG002_BC3C_24H:A10', 'ASG002_BC3C_24H:A11', 'ASG002_BC3C_24H:A19', 'ASG002_BC3C_24H:A20', 'ASG002_BC3C_24H:A21', 'ASG002_BC3C_24H:B10', 'ASG002_BC3C_24H:B11', 'ASG002_BC3C_24H:B14', 'ASG002_BC3C_24H:B15', 'ASG002_BC3C_24H:C13', 'ASG002_BC3C_24H:C14', 'ASG002_BC3C_24H:C15', 'ASG002_BC3C_24H:E19', 'ASG002_BC3C_24H:E20', 'ASG002_BC3C_24H:E21', 'ASG002_BC3C_24H:E22', 'ASG002_BC3C_24H:E23', 'ASG002_BC3C

In [80]:
# save doses and perturbation matrix
inhib_conc_df.to_csv(os.path.join(out_dir, "inhib_conc_annotated.csv"))
ic50_df.to_csv(os.path.join(out_dir, "ic50_annotated.csv"))
# gamma_df.to_csv(os.path.join(out_dir, "gamma_annotated.csv"))
pert_df.to_csv(os.path.join(out_dir, "pert_annotated.csv"))

In [81]:
# save log fold change L1000 data
Data_norm_df.to_csv(os.path.join(out_dir, "L1000_Data_norm_data.csv"))

In [82]:
# save R_global for DPDs
R_global_DPD_df.to_csv(os.path.join(out_dir, "R_global_DPDonly_annotated.csv"))